<a href="https://colab.research.google.com/github/AristidesAntonioOrellanaZelaya/etl-proyecto-bi/blob/main/Notebooks/Empleo_turistico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Cargar archivo
df_empleo = pd.read_csv("https://raw.githubusercontent.com/AristidesAntonioOrellanaZelaya/etl-proyecto-bi/refs/heads/main/Data/fact_empleo_turistico.csv")

print("Registros originales:", len(df_empleo))

df_empleo.head()

Registros originales: 48


,anio,mes,empleos,sector
0,2021,1,61848,Turismo
1,2021,2,65836,Turismo
2,2021,3,64123,Turismo
3,2021,4,67691,Turismo
4,2021,5,55155,Turismo


Revisar estructura

In [2]:
df_empleo.info()

df_empleo.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   anio     48 non-null     int64 
 1   mes      48 non-null     int64 
 2   empleos  48 non-null     int64 
 3   sector   48 non-null     object
dtypes: int64(3), object(1)
memory usage: 1.6+ KB


,0
anio,0
mes,0
empleos,0
sector,0


**Transformar**

Eliminar duplicados

In [3]:
print(
    "Duplicados:",
    df_empleo.duplicated().sum()
)

df_empleo.drop_duplicates(inplace=True)

Duplicados: 0


Validar año

In [4]:
df_empleo = df_empleo[
    (df_empleo['anio'] >= 2021)
    &
    (df_empleo['anio'] <= 2024)
]

Validar mes

In [5]:
df_empleo = df_empleo[
    (df_empleo['mes'] >= 1)
    &
    (df_empleo['mes'] <= 12)
]

Limpiar sector

In [6]:
df_empleo['sector'] = (
    df_empleo['sector']
    .astype(str)
    .str.strip()
    .str.title()
)

Validar empleos

In [7]:
df_empleo = df_empleo[
    df_empleo['empleos'] > 0
]

Crear fecha analítica

In [8]:
df_empleo['fecha'] = pd.to_datetime(
    df_empleo['anio'].astype(str)
    + '-'
    + df_empleo['mes'].astype(str)
    + '-01'
)

Crear trimestre

In [9]:
df_empleo['trimestre'] = (
    df_empleo['fecha']
    .dt.quarter
)

Crear crecimiento mensual de empleo

In [10]:
df_empleo = df_empleo.sort_values(
    by=['anio', 'mes']
)

df_empleo['crecimiento_empleo_pct'] = (
    df_empleo['empleos']
    .pct_change()
    * 100
).round(2)

Clasificar nivel de empleo

In [11]:
df_empleo['nivel_empleo'] = np.where(
    df_empleo['empleos'] < 50000,
    'Bajo',
    np.where(
        df_empleo['empleos'] < 100000,
        'Medio',
        'Alto'
    )
)

Reemplazar el primer valor nulo del crecimiento

In [12]:
df_empleo['crecimiento_empleo_pct'] = (
    df_empleo['crecimiento_empleo_pct']
    .fillna(0)
)

Validación

In [13]:
print(df_empleo.info())

df_empleo.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   anio                    48 non-null     int64         
 1   mes                     48 non-null     int64         
 2   empleos                 48 non-null     int64         
 3   sector                  48 non-null     object        
 4   fecha                   48 non-null     datetime64[ns]
 5   trimestre               48 non-null     int32         
 6   crecimiento_empleo_pct  48 non-null     float64       
 7   nivel_empleo            48 non-null     object        
dtypes: datetime64[ns](1), float64(1), int32(1), int64(3), object(2)
memory usage: 2.9+ KB
None


,anio,mes,empleos,fecha,trimestre,crecimiento_empleo_pct
count,48.000000,48.000000,48.000000,48,48.000000,48.000000
mean,2022.500000,6.500000,61489.979167,2022-12-16 05:00:00,2.500000,0.907292
min,2021.000000,1.000000,52224.000000,2021-01-01 00:00:00,1.000000,-23.110000
25%,2021.750000,3.750000,56324.000000,2021-12-24 06:00:00,1.750000,-8.225000
50%,2022.500000,6.500000,62243.500000,2022-12-16 12:00:00,2.500000,-1.575000
75%,2023.250000,9.250000,66672.000000,2023-12-08 18:00:00,3.250000,8.805000
max,2024.000000,12.000000,69704.000000,2024-12-01 00:00:00,4.000000,31.300000
std,1.129865,3.488583,5581.071195,NaN,1.129865,13.745533


**Carga**

In [14]:
df_empleo.to_csv(
    'dw_fact_empleo_turistico.csv',
    index=False
)

print("ETL completado correctamente")

ETL completado correctamente
